In [1]:
import os
import glob
import time
import random
import yaml
import torch

# === 1. VERIFY GPU ===
assert torch.cuda.is_available(), "ERROR: No GPU detected! Select GPU T4 x2 or P100."
num_gpus = torch.cuda.device_count()
device_arg = "0,1" if num_gpus > 1 else "0"
batch_size = 32 if num_gpus > 1 else 16
print(f"GPU Active: {num_gpus} GPU(s) | device='{device_arg}' | batch={batch_size}")

# === 2. LOCATE PHASE 3 best.pt ===
best_pts = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
assert best_pts, "ERROR: No .pt weights found in /kaggle/input!"
phase3_matches = [p for p in best_pts if 'phase3' in p.lower() or 'roboflow_mixed' in p.lower()]
start_weights = phase3_matches[0] if phase3_matches else best_pts[-1]
print(f"Initializing Phase 5 from weights: {start_weights}")

# === 3. AUTO-DISCOVER DEEPFASHION2, ROBOFLOW, & ANGLED/FOLDED V2 ===
all_train = sorted(glob.glob('/kaggle/input/**/images/train/*.*', recursive=True))
all_val = sorted(glob.glob('/kaggle/input/**/images/val/*.*', recursive=True))

df_train_all = [p for p in all_train if 'train_part_' in p]
df_val_all = [p for p in all_val if 'val_part_' in p]

angled_train = [p for p in all_train if 'angled' in p.lower() or 'folded' in p.lower()]
angled_val = [p for p in all_val if 'angled' in p.lower() or 'folded' in p.lower()]

rf_train = [p for p in all_train if 'train_part_' not in p and 'angled' not in p.lower() and 'folded' not in p.lower()]
rf_val = [p for p in all_val if 'val_part_' not in p and 'angled' not in p.lower() and 'folded' not in p.lower()]

print(f"Angled/Folded v2: {len(angled_train):,} train | {len(angled_val):,} val")
print(f"Roboflow Phase 3: {len(rf_train):,} train | {len(rf_val):,} val")
print(f"DeepFashion2:     {len(df_train_all):,} train | {len(df_val_all):,} val")

assert len(angled_train) >= 30 and len(rf_train) >= 1000 and len(df_train_all) >= 4000, "Missing dataset files!"

# === 4. BALANCED PHASE 5 MANIFEST (~7,258 TRAIN IMAGES) ===
rng = random.Random(2027)
df_train_sample = rng.sample(df_train_all, 4000)
df_val_sample = rng.sample(df_val_all, 800)

mixed_train = (angled_train * 10) + (rf_train * 2) + df_train_sample
mixed_val = (angled_val * 5) + rf_val + df_val_sample
rng.shuffle(mixed_train)

with open('/kaggle/working/train_phase5.txt', 'w') as f:
    f.write('\n'.join(mixed_train) + '\n')
with open('/kaggle/working/val_phase5.txt', 'w') as f:
    f.write('\n'.join(mixed_val) + '\n')

print(f"Phase 5 manifests: {len(mixed_train):,} train | {len(mixed_val):,} val")

# === 5. CONFIGURE data_phase5.yaml ===
yaml_path = '/kaggle/working/data_phase5.yaml'
data_yaml_content = {
    'path': '/kaggle/working',
    'train': 'train_phase5.txt',
    'val': 'val_phase5.txt',
    'names': {
        0: 'short sleeve top',
        1: 'long sleeve top',
        2: 'short sleeve outwear',
        3: 'long sleeve outwear',
        4: 'vest',
        5: 'sling',
        6: 'shorts',
        7: 'trousers',
        8: 'skirt',
        9: 'short sleeve dress',
        10: 'long sleeve dress',
        11: 'vest dress',
        12: 'sling dress'
    }
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

# === 6. FINE-TUNE WITH ROTATION & PERSPECTIVE AUGMENTATION ===
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO(start_weights)

print("\nStarting Phase 5 Angled & Folded Fine-Tuning (20 epochs)...")
start_time = time.time()

model.train(
    data=yaml_path,
    epochs=20,
    patience=7,
    imgsz=640,
    batch=batch_size,
    device=device_arg,
    workers=4,
    lr0=0.0008,
    lrf=0.01,
    cos_lr=True,
    degrees=15.0,
    shear=5.0,
    perspective=0.0005,
    mosaic=1.0,
    mixup=0.1,
    scale=0.5,
    project='/kaggle/working/fitloop_runs',
    name='phase5_angled_wardrobe',
    save=True,
    save_period=1,
    exist_ok=True,
    verbose=True
)

elapsed_mins = (time.time() - start_time) / 60
print(f"\nPHASE 5 COMPLETE in {elapsed_mins:.2f} minutes!")
print("New best weights: /kaggle/working/fitloop_runs/phase5_angled_wardrobe/weights/best.pt")

GPU Active: 2 GPU(s) | device='0,1' | batch=32
Initializing Phase 5 from weights: /kaggle/input/datasets/spoortimorabad/fitloop-phase3-weights/best_phase3.pt
Angled/Folded v2: 63 train | 3 val
Roboflow Phase 3: 1,404 train | 254 val
DeepFashion2:     191,958 train | 32,153 val
Phase 5 manifests: 7,438 train | 1,069 val
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

Starting Phase 5 Angled & Folded Fine-Tuning (20 epochs)...
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       2/20      3.54G      1.301       1.49    0.02143         55        640: 100% ━━━━━━━━━━━━ 233/233 3.1it/s 1:15
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 3.5it/s 4.9s
                   all       1069       1902      0.724      0.624      0.715      0.439

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/20      3.54G      1.253      1.419    0.02076         40        640: 100% ━━━━━━━━━━━━ 233/233 3.2it/s 1:13
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 3.0it/s 5.6s
                   all       1069       1902       0.64      0.697      0.677      0.443

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/20      3.55G      1.247      1.401    0.02107         31        640: 100% ━━━━━━━━━━━

import os
import glob
import time
import random
import yaml
import torch

# === 1. VERIFY GPU ===
assert torch.cuda.is_available(), "ERROR: No GPU detected! Select GPU T4 x2 or P100."
num_gpus = torch.cuda.device_count()
device_arg = "0,1" if num_gpus > 1 else "0"
batch_size = 32 if num_gpus > 1 else 16
print(f"GPU Active: {num_gpus} GPU(s) | device='{device_arg}' | batch={batch_size}")

# === 2. LOCATE PHASE 1 best.pt IN /kaggle/input ===
best_pt_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
assert best_pt_candidates, "ERROR: Upload your Phase 1 best.pt into /kaggle/input first!"
phase1_weights = best_pt_candidates[0]
print(f"Found Phase 1 weights: {phase1_weights}")

# === 3. SAMPLE 35,000 TRAIN / 5,000 VAL IMAGES ===
print("\nScanning /kaggle/input for FitLoop images...")
all_train_imgs = sorted(glob.glob('/kaggle/input/**/images/train/*.jpg', recursive=True))
all_val_imgs = sorted(glob.glob('/kaggle/input/**/images/val/*.jpg', recursive=True))

assert len(all_train_imgs) >= 35000, "Dataset incomplete!"
assert len(all_val_imgs) >= 5000, "Validation set incomplete!"

rng = random.Random(101)
subset_train = sorted(rng.sample(all_train_imgs, 35000))
subset_val = sorted(rng.sample(all_val_imgs, 5000))

with open('/kaggle/working/train_35k.txt', 'w') as f:
    f.write('\n'.join(subset_train) + '\n')
with open('/kaggle/working/val_5k.txt', 'w') as f:
    f.write('\n'.join(subset_val) + '\n')

print(f"Saved Phase 2 manifests: {len(subset_train):,} train | {len(subset_val):,} val")

# === 4. CONFIGURE data_35k.yaml ===
yaml_path = '/kaggle/working/data_35k.yaml'
data_yaml_content = {
    'path': '/kaggle/working',
    'train': 'train_35k.txt',
    'val': 'val_5k.txt',
    'names': {
        0: 'short sleeve top',
        1: 'long sleeve top',
        2: 'short sleeve outwear',
        3: 'long sleeve outwear',
        4: 'vest',
        5: 'sling',
        6: 'shorts',
        7: 'trousers',
        8: 'skirt',
        9: 'short sleeve dress',
        10: 'long sleeve dress',
        11: 'vest dress',
        12: 'sling dress'
    }
}
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

# === 5. FINE-TUNE FROM PHASE 1 best.pt ===
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO(phase1_weights)

print("\nStarting Phase 2 Fine-Tuning (25 epochs, lr0=0.002, cos_lr=True)...")
start_time = time.time()

model.train(
    data=yaml_path,
    epochs=25,
    patience=8,
    imgsz=640,
    batch=batch_size,
    device=device_arg,
    workers=4,
    lr0=0.002,
    lrf=0.01,
    cos_lr=True,
    project='/kaggle/working/fitloop_runs',
    name='phase2_35k_yolo26n',
    save=True,
    save_period=1,
    exist_ok=True,
    verbose=True
)

elapsed_mins = (time.time() - start_time) / 60
print(f"\nPHASE 2 COMPLETE in {elapsed_mins:.2f} minutes!")
print("New best weights: /kaggle/working/fitloop_runs/phase2_35k_yolo26n/weights/best.pt")